In [ ]:
from google.colab import drive

drive.mount('/content/drive')


# Geometry-V1 D1 independent confirmation handoff

Run this notebook once, top to bottom. It is a Drive-only, CPU-or-GPU-neutral launcher for the bounded D1 artifact runner; the notebook neither loads a model nor inspects ZIP records. D1 retains `science_denominator=0` and does not make a route, detector, method, or scientific success claim.


In [ ]:
import json
import os
import pathlib
import subprocess
import time

SOURCE_D01_ARTIFACT_EXACT = 'ccfb7bcefbb18f9812a4e800bbea18b91b031ebb'
D1_RUNNER_EXACT = '906478a04334118c1fd71996e38ab905bea6d35a'
SOURCE_RUN_ID = 'geometry-v1-qk-d01-ccfb7bcefbb1'
SOURCE_PROTOCOL = 'geometry-v1-qk-d01-artifact-selection-v1'
SOURCE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D01/Geometry-V1-QK-D01-ccfb7bcefbb1-20260827T083601Z')
OUTPUT_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D1')
RUNNER_PATH = 'experiments/run_geometry_v1_qk_d1_independent_confirmation_operational.py'
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D1 '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D1_FAILURE '
MAX_CONTROL_BYTES = 1024
D1_FAILED = False
RUNNER_ATTEMPTED = False

def public_error_class(error):
    return 'filesystem_error' if isinstance(error, OSError) else 'checkout_error'

def fail_closed(stage, error):
    global D1_FAILED
    if not D1_FAILED:
        D1_FAILED = True
        print('CEGWM_GEOMETRY_V1_QK_D1_HANDOFF_FAILURE ' + json.dumps({'stage': stage, 'error_class': public_error_class(error)}, sort_keys=True, separators=(',', ':')))


In [ ]:
if not D1_FAILED:
    try:
        if not SOURCE_ROOT.is_dir(): raise RuntimeError('source artifact unavailable')
        repo = pathlib.Path('/content/Geometry-V1-D1')
        if repo.exists(): raise RuntimeError('fresh checkout path already exists')
        subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/RICHAAARC/CEG-WM.git', str(repo)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['git', 'checkout', '--detach', D1_RUNNER_EXACT], cwd=repo, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        execution_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
        checkout_clean = not subprocess.run(['git', 'status', '--porcelain'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
        if execution_commit != D1_RUNNER_EXACT or not checkout_clean: raise RuntimeError('runner checkout identity mismatch')
        runner_path = repo / RUNNER_PATH
        if not runner_path.is_file(): raise RuntimeError('runner path unavailable')
        source_d01_artifact_identity = {'run_id': SOURCE_RUN_ID, 'execution_exact': SOURCE_D01_ARTIFACT_EXACT, 'protocol': SOURCE_PROTOCOL, 'path': str(SOURCE_ROOT)}
        runner_execution_identity = {'commit': execution_commit, 'clean': checkout_clean}
        print('CEGWM_GEOMETRY_V1_QK_D1_CHECKOUT ' + json.dumps({'source_d01_artifact_identity': source_d01_artifact_identity, 'runner_execution_identity': runner_execution_identity}, sort_keys=True, separators=(',', ':')))
    except BaseException as error:
        fail_closed('checkout', error)


In [ ]:
if not D1_FAILED:
    try:
        if RUNNER_ATTEMPTED: raise RuntimeError('runner already attempted')
        RUNNER_ATTEMPTED = True
        OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        session_utc = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
        run_dir = OUTPUT_ROOT / ('Geometry-V1-QK-D1-' + execution_commit[:12] + '-' + session_utc)
        if run_dir.exists(): raise RuntimeError('create-only D1 output already exists')
        control_read, control_write = os.pipe()
        command = ['python', str(runner_path), '--repo-root', str(repo), '--expected-exact', execution_commit, '--source-root', str(SOURCE_ROOT), '--output-root', str(run_dir), '--control-fd', str(control_write)]
        process = subprocess.Popen(command, cwd=repo, pass_fds=(control_write,), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write)
        try:
            runner_rc = process.wait(timeout=7200)
            control_line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        finally:
            os.close(control_read)
        if len(control_line) > MAX_CONTROL_BYTES: raise RuntimeError('bounded control exceeded')
        if control_line.startswith(SUCCESS_PREFIX.encode('ascii')):
            control = json.loads(control_line[len(SUCCESS_PREFIX):])
        elif control_line.startswith(FAILURE_PREFIX.encode('ascii')):
            control = json.loads(control_line[len(FAILURE_PREFIX):])
        else:
            control = {'status': 'unavailable'}
        terminal = {'runner_rc': runner_rc, 'control': control, 'drive_directory': str(run_dir), 'source_d01_artifact_identity': source_d01_artifact_identity, 'runner_execution_identity': runner_execution_identity}
        print('CEGWM_GEOMETRY_V1_QK_D1_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
        if runner_rc != 0 or control.get('status') != 'success': raise RuntimeError('D1 runner failed')
    except BaseException as error:
        fail_closed('runner', error)
